# Rebuilding the joint decomposition, by hand

This notebook walks you through reimplementing the v3 joint decomposition **from scratch**,
using only `numpy`/`scipy` and the primitives of the `tracking` package. Every section
explains one piece — the model term it implements, the exact operations, and which existing
primitive to compare your implementation against. The code cells are **empty on purpose**:
you write them.

Companions, in order of usefulness:

* the **"Joint Noise Model" explainer artifact** — full derivations from probability
  densities to the algorithm box (§8–§14 are the sections this notebook implements);
* `src/tracking/AGENTS.md` — the package map: four layers, the Stage contract, and the
  primitive catalog (§1 is the stage/combinator/recipe table);
* `docs/vk-decompose-v3-design.md` and `docs/experiments/vk-decomposition.md` — the design
  record and the campaign log with every measured number quoted below.

**Ground rules learned the hard way** (each cost this project a wrong conclusion):
verify comb removal with the *full order-cell profile at all offsets* and the
*absolute excess* — never narrow slots, never peak-over-median depth (blind to broadened
lines); render the spectrogram and **look at it** before believing any instrument; and
quote phase-noise statistics in *increment* units (Hz rms), never cumulative rad rms.


## 1 · The noise model, in increments

Everything is discrete. Audio sample $n$ at rate $f_s$ (spacing $\Delta = 1/f_s$); an
envelope grid $j$ with stride $L$ samples ($f_s = 32\,$kHz, $L = 320$, $f_e = 100\,$Hz in
production). $\mathrm{lin}(\cdot)$ = linear interpolation from the envelope grid to audio.

**The carrier is a cumulative sum of the label rates** $\hat f_r[n]$ (rev/s):

$$\hat\varphi_r[n] = 2\pi\Delta \sum_{u\le n} \hat f_r[u]$$

**The noise terms are rate processes whose *increments* are the random variables.**
Shaft rate error $\delta\!f_r[j]$ (per rotor, felt by harmonic $k$ as $k\,\delta\!f_r$) and
per-track wobble $\delta\nu_{r,k}[j]$, with priors on their *steps only*:

$$\delta\!f_r[j] - \delta\!f_r[j{-}1] \sim \mathcal N(0, q_\theta/f_e), \qquad
\delta\nu_{r,k}[j] - \delta\nu_{r,k}[j{-}1] \sim \mathcal N(0, q_\varepsilon(k)/f_e)$$

The phase corrections are their cumulative sums, $\theta_r = \frac{2\pi}{f_e}\sum \delta\!f_r$
and $\psi_{r,k} = \frac{2\pi}{f_e}\sum \delta\nu_{r,k}$ — so the *second difference* of each
phase correction is white, which is exactly the $\lVert D_2 x\rVert^2$ penalty every solver
below applies. Nothing prices position: constant offsets and constant frequency errors are
free (the $D_2$ null space), phases wander unboundedly, harmonics decohere. That is the
measured physics, not a modeling convenience.

**Observation**, with complex envelopes $A_{r,k,c}[j]$ (magnitude = loudness, free static
angle = initial phase per mic) and a smooth-log-PSD floor $n_c$:

$$y_c[n] = \sum_{r,k} \mathrm{Re}\Big[\mathrm{lin}(A_{r,k,c})[n]\,
e^{\,i\,k(\hat\varphi_r[n] + \mathrm{lin}(\theta_r)[n]) + i\,\mathrm{lin}(\psi_{r,k})[n]}\Big] + n_c[n]$$

The MAP objective (explainer §10) is the whitened misfit $+\log S$ rent, plus one
increment penalty per process, plus amplitude and floor smoothness. The algorithm you will
build is block-coordinate descent on it: `floor → (solve → split → floor)×(I−1) → solve`.

The strengths are parameterized by −3 dB bandwidth through one shared calibration:
$\lambda(\mathrm{bw}) = (2\sin(\pi\,\mathrm{bw}/f_e))^{-4}$. Shipped values:
$\mathrm{bw}_\theta = 1.5$ Hz, $\mathrm{bw}_\psi(k) = \mathrm{clip}(0.6k,\,1.5,\,8)$ Hz,
amplitude bands 1.9–2.9 Hz from the v2 schedule.


## 2 · Orientation: the package in four layers

`src/tracking` is arranged so you only ever need two files to navigate:

| Layer | Module | Role |
|---|---|---|
| front door | `top.py` | **every** Stage + every shipped variant as a named composition |
| ladders | `pipelines.py` | calibrated multi-step algorithms |
| cores | 13 modules | one algorithm each, arrays in / arrays out |
| primitives | `dsp.py` | the shared transforms |

The Stage contract: `Stage: td.Frame -> td.Frame`, composed with `pipeline(a, b, ...)`;
combinators `iterate(stage, n)` and `windowed(inner, window_s=, hop_s=)`; diagnostics
accumulate in `meta["tracking"]`. The v3 blocks are `vk_solve_stage` / `phase_split_stage` /
`floor_stage` seeded by `joint_init_stage`, and the reference recipe is
`tracking.joint_solve_window`. You will rebuild each block as plain array code first, check
it against the core functions, and only then (optionally) wrap yours as Stages.

**Setup cell.** Import `numpy`, `scipy`, `tracking`, `tdseries as td`. Load one recording
with its refined labels — the DREGON hover `free-flight_nosource_room1` is the reference:
`scripts/vk_decompose.py` exposes `get_recording` (pass `sr=32000` **explicitly** — it is a
def-time default and module globals do nothing), or use `plots.explore.pick`. Slice a 12 s
window, keep 2 mics to stay light. Everything below runs comfortably on a laptop at this
size; full recordings go to the cluster.


## 3 · The carrier

Build $\hat\varphi_r[n] = 2\pi\Delta\,\mathrm{cumsum}(\hat f_r)$ for each rotor from the
label rates, and the per-track phases $\Phi_m = k_m \hat\varphi_{r_m}$ for tracks
$m = (r, k)$, $k = 1 \dots k_{\mathrm{hi}}$ (cap lines at $0.375 f_s$).

*Check:* `tracking.joint_init_stage` (or `joint_state`) builds exactly this carrier; the
project's rule that the carrier is *held* in the solver state, never re-derived, exists
because re-interpolating it perturbs phases at the $10^{-7}$ rad level and breaks
bit-reproducibility.


## 4 · Block A — the banded envelope solve

The heart. For the model above, with everything but the envelopes frozen, the MAP problem in
$A$ is a penalized least squares whose normal equations are **diagonal in envelope time**
(the brick-wall decimation makes inner products local). The operations, exactly:

1. **Demodulate + decimate**: $z_{m,c}[j] = \mathcal D_L\{y_c[n]\,e^{-i\Phi_m[n]}\}[j]$
   where $\mathcal D_L$ = FFT → keep bins $|f| \le 0.45 f_e$ → inverse FFT at the short
   length → divide by $L$ (a zoom-IFFT; no filter design).
2. **Cross-kernels** for tracks that ever come within the coupling bandwidth (group them
   with a union–find): $g_{mn}[j] = \mathcal D_L\{e^{i(\Phi_n - \Phi_m)}\}[j]$.
3. **Normal equations** per group ($u \equiv 1$, $w$ = validity+edge taper for now —
   whitening arrives in §6):
   $$\big(\mathrm{diag}(w u^2) + \rho_m^2 D_2^\top D_2\big) x_m
     + \sum_{n\ne m} \mathrm{diag}(w\,u_m u_n\, g_{mn})\, x_n
     = \mathrm{diag}(w u^2)\, z_m$$
   Interleave unknowns time-major → Hermitian **banded**, $2g$ superdiagonals.
4. **Solve** with `scipy.linalg.cholesky_banded` + `cho_solve_banded`; on a
   non-positive-definite report, retry with the diagonal × $(1+10^{-6})$, then
   $(1+10^{-4})$.
5. **Reconstruct**: comb $= \sum_m \mathrm{Re}[\mathrm{lin}(x_m)[n]\, e^{i\Phi_m[n]}]$;
   residual $=$ audio − comb, exactly.

Start with a *single rotor, k ≤ 10, no coupling* to get the machinery right, then add the
groups. The edge taper: fade $w$ over the first/last $\min(8, J/4)$ envelope samples.

*Check:* `tracking.vk_envelopes` (the core; your envelopes should match to solver
tolerance), `tracking.vk_stage` (the Stage wrapper), `tracking.reconstruct`.


## 5 · Look, then measure

Render the triptych — original / your comb / your residual — as 0–8 kHz spectrograms with a
**shared** palette (percentiles of the *original*). The eyeball check is mandated policy in
this project: it has caught what three instruments missed.

Then the honest instrument. Order-cell profile: Hann STFT (8192 points), per frame
interpolate the power onto the order grid $f / \hat f_r(t)$ at 0.005-order steps, average
unit cells per band (k1–9 / 10–24 / 25–49 / 50–80), **detrend with a running median over
one order** (without it, the sloped floor fakes half-order peaks — that artifact cost this
project a retraction). Report per band: the **absolute excess** (summed peak − median, in
power units) of your residual vs the original, as *excess retained %*. Depth (peak/median)
is blind to broadened lines — never gate on it.

*Check:* `tracking.order_cell_bands`, `tracking.stft_power`. Expect your plain (unwhitened,
uncorrected) solve to clean k1–9 well and leave most of k≥10 — that *is* the v2 story, and
the reason the rest of this notebook exists.


## 6 · Block C — the floor between the lines, and whitening

The floor $S_c(f, t)$: Hann STFT ($n_{\mathrm{fft}} = 4096$, hop 2048, PSD scale
$1/(f_s \sum \mathrm{hann}^2)$); per frame **mask** every predicted line to
$\pm\,\mathrm{clip}(3 \cdot 0.6k,\ 10\ \mathrm{Hz},\ 0.45\,\hat f_r)$ — several linewidths,
because strong lines' skirts lift the floor far out; pool unmasked power into 4 time blocks;
smooth each block's log spectrum with a 9-bin moving median then a **cepstral lift** (DCT,
keep 40 coefficients, inverse). Masking is not optional: an unmasked fit rises under the
lines and tells block A not to bother — a self-sealing failure.

Whitening weights for block A: $u_m[j] = e^{-\frac12 \log S(k_m \hat f_{r_m}(t_j),\, t_j)}$,
clamped to ±15 dB, normalized to geometric mean 1 (whitening sets *relative* trust only) —
and carry $\overline{u_m^2}$ into $\rho_m^2$ too, so a down-weighted track does not silently
narrow its band (skipping this cost 8 dB of residual comb on the synthetic fixture).

*Check:* `tracking.masked_smooth_psd`, `tracking.whiten_weights`, `tracking.floor_stage`.


## 7 · Block B — reading the walks off the solved phases

Input: the solved bank $x[c, m, j]$ against the current carrier. Its angle *is* the phase
error that is left. Operations:

1. Combine channels $v_m = \sum_c x[c,m,:]$; $a_m = \mathrm{unwrap}(\arg v_m)$, remove each
   track's mean (the constant is the envelope's gauge).
2. Gates: concentration $\kappa_m = |\mathrm{mean}_j\, e^{i \Delta \arg v_m}|$ (wrapped
   first differences) ≥ 0.5, no unwrap step reaching $\pi$, and — the annealing ladder —
   $k_m \le k_{\mathrm{trust}}$, because low harmonics see the shaft error only $k$-fold and
   are unambiguous first.
3. Shaft, weights $w_m = k_m^2 \kappa_m^2$: the $k^2$-weighted mean of $a_m / k_m$ over
   trusted tracks, smoothed by the 1-D Whittaker–Henderson solve
   $x = (\mathrm{diag}(w^{\mathrm{tap}}) + \lambda D_2^\top D_2)^{-1} \mathrm{diag}(w^{\mathrm{tap}})\,y$
   at $\lambda(\mathrm{bw}_\theta)$ — with the **edge taper in the data weight** so the
   estimate extrapolates over the faded window ends (skipping this puts ~0.5 rev/s of
   fabricated rate at the seams). Then a per-rotor pass on the leftovers.
4. Per-track $\psi_m$ = the same smoother on $a_m - k_m \theta_{r_m}$ at
   $\lambda(\mathrm{bw}_\psi(k_m))$, strong tracks only, from round 2 on.
5. **Fold**: $\theta \mathrel{+}= \Delta\theta$ into the carrier (exact — next round's
   demodulation uses it); $\psi$ by rotating the demodulated data
   ($z_m e^{-i\psi_m}$, cross-kernels by $e^{i(\psi_n-\psi_m)}$).

*Check:* `tracking.split_phases` (compare $\theta$, $\psi$, and the diagnostics),
`tracking.wh_smooth` / `wh_lambda`, `tracking.phase_split_stage`.


## 8 · The alternation

Compose your three blocks exactly as the algorithm box (explainer §14):

```
S ← FLOOR(y)
for t in 1..3:
    x, resid ← SOLVE(y; carrier+θ, ψ, S)
    if t == 3: break
    θ, ψ  +=  SPLIT(x; k_trust = (3, 12, 80)[t], with-ψ iff t ≥ 2)
    S ← FLOOR(resid)
```

Per round, record the readings: track/residual energy shares, whitened flatness
(spectral flatness of $P^{\mathrm{res}}/S$), excess retained per band. You should see the
ladder work: round 1 cleans low k, each fold shrinks the remaining phase error, higher k
becomes trustable. Why this descends one objective and in what sense it converges (it is
hard EM) — explainer §13.

*Check:* the recipe `tracking.joint_solve_window`, or the Stage form
`pipeline(joint_init_stage(...), floor_stage(...), iterate(pipeline(vk_solve_stage(...),
phase_split_stage(...), floor_stage(...)), 2), vk_solve_stage(...))`. The regression
fixture `tests/tracking/fixtures/joint_v3b_reference.npz` pins the reference numbers at
1e-10 — your implementation should land close on the same synthetic input
(`tests/tracking/_joint_fixture.py` builds it).


## 9 · A full recording

12 s windows, 9 s hop, each window through your loop independently, then stitch: raised-
cosine cross-fade over the 3 s overlaps for envelopes, corrections and residual; average the
per-window $\dot\theta / 2\pi$ into a global label correction. Verify with the gates:
excess retained per band, whitened flatness, the triptych. The production reference outputs
(v2, v3, v3b for all three recordings) are on R2 under `artifacts/vk-decompose-*/` for
comparison; the campaign log has every number.

*Check:* `tracking.windowed`, `tracking.stitch_windows`, `tracking.window_geometry`;
driver: `scripts/vk_decompose.py --joint` (thin glue over exactly what you just built).


## 10 · Where the method actually stands, and what is yours to build

Honest state of the reference implementation (v3b, from the campaign log): k1–9 is clean
(≤ 4 % of comb excess retained), but **30–37 % of comb-locked excess remains at k10–24 and
56–88 % at k25–80** — the per-track bandwidth cap (8 Hz) truncates the fast increments of
walks whose linewidth grows as $0.6k$ Hz, and where lines sit closer than their linewidth no
coherent bandwidth can work at all. Three designed-but-unbuilt levers, in difficulty order —
these are the natural continuation of this notebook, each specified in the campaign log:

1. **Spacing-limited bandwidth**: $\mathrm{bw}_\psi = \min(0.6k,\ 0.4 \times$ distance to
   the nearest other predicted line$)$, no absolute cap.
2. **The regime-3 statistical split**: where coherent tracking is impossible, per STFT frame
   measure band power $P$ in $\min(0.6k,\ \mathrm{spacing})$ around the corrected carrier,
   and move the Wiener excess $\max(0,\ 1 - S B / P)$ into an explicit *stochastic comb*
   channel — keeping the exact four-way identity coherent + stochastic + tones + residual
   = input. This is what the model's own theory (explainer §17) says the merged regime
   requires; it is the part v3b never implemented, and it governs three quarters of the
   spectrogram.
3. **The foreign-tone dictionary**: constant-frequency VK tracks seeded from
   `tracking.residual_tones` (DREGON's 489 Hz class), their own output channel, so the
   broadband floor can finally be as smooth as the model demands.

Also open: FLY rotor 0 keeps a genuine 2.2 dB integer-order leftover at k1–9 on both
flights (rig-systematic — that rotor's labels or acoustics; the ψ-floor lever measured
inert). And one standing caveat for anything you train downstream: v2/v3b-derived
"broadband" targets are comb-contaminated above k10 — do not train on them.


## References

* The **Joint Noise Model** explainer (derivations §8–§14, convergence §13, regime map
  §17, probe verdicts §18, reproduction commands §19) — artifact `1c8c18a5`, 📐.
* The **Anatomy of a Drone Recording** demo (spectrograms + audio, v3b, honesty notes) —
  artifact `a7b9a27a`, 🎼.
* `src/tracking/AGENTS.md` — package map + primitive catalog.
* `docs/vk-decompose-v3-design.md`, `docs/experiments/vk-decomposition.md` — design +
  campaign log (every number, every retraction, every instrument audit).
